In [1]:
import json
import time
import re
from collections import Counter
from IPython.display import display, Markdown, HTML
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# ============================================================
#  CONFIGURATION
# ============================================================
MODEL = "qwen2.5:1.5b"  #qwen2.5:1.5b -> alternate model

llm = ChatOllama(model=MODEL)

# ============================================================
#  HELPER FUNCTIONS
# ============================================================

def build_messages(message_dicts):
    """Convert role/content dicts into LangChain message objects."""
    type_map = {"system": SystemMessage, "user": HumanMessage, "assistant": AIMessage}
    return [type_map[m["role"]](content=m["content"]) for m in message_dicts]


def chat(messages, show=True, **kwargs):
    """Send messages to the model and return the response text."""
    _llm = ChatOllama(model=MODEL, **kwargs) if kwargs else llm
    lc_messages = build_messages(messages)
    start = time.time()
    response = _llm.invoke(lc_messages)
    elapsed = time.time() - start
    content = response.content
    if show:
        display(Markdown(content))
        print(f"\n⏱️ {elapsed:.2f}s | {len(content)} chars")
    return content


def show_messages(messages):
    """Pretty-print the message list."""
    colors = {"system": "#e74c3c", "user": "#3498db", "assistant": "#2ecc71"}
    html = ""
    for msg in messages:
        role = msg["role"]
        color = colors.get(role, "#888")
        preview = msg["content"][:400] + ("..." if len(msg["content"]) > 400 else "")
        html += (
            f'<div style="margin:6px 0;padding:8px 12px;border-left:4px solid {color};'
            f'background:#1e1e1e;border-radius:4px;">'
            f'<strong style="color:{color};text-transform:uppercase;">{role}</strong>'
            f'<br><span style="color:#ccc;white-space:pre-wrap;">{preview}</span></div>'
        )
    display(HTML(html))


def section_header(title, subtitle=""):
    html = f"""
    <div style="background:linear-gradient(135deg,#1a1a2e,#16213e);padding:16px 20px;
                border-radius:8px;border-left:5px solid #e94560;margin:20px 0;">
      <h2 style="color:#e94560;margin:0;">{title}</h2>
      <p style="color:#aaa;margin:6px 0 0;">{subtitle}</p>
    </div>"""
    display(HTML(html))


print(f"✅ Using model: {MODEL}")

✅ Using model: qwen2.5:1.5b


## 1. Chain of Thought (CoT) Prompting

In [4]:
# ZERO SHOT

problem = (
    "A store sells shirts for $8 each and pants for $10. " 
    "If Ajay buys 5 shirts and 3 pants, then uses 20% discount coupon. " 
    "how much does he pay in total?"    
)
print("=" * 60)
print("STANDARD PROMPT")
print("=" * 60)
messages_std = [
    {"role": "system", "content": "You are helpful math assistant. Give only final answer."},
    {"role": "user", "content": problem }
]

show_messages(messages_std)
_ = chat(messages_std, temperature = 0.0)

print("=" * 60)
print("PROMPT WITH COT")
print("=" * 60)

messages_cot = [
    {"role": "system", "content": "You are helpful math assistant. Give only final answer."},
    {"role": "user", "content": problem + "\n\n Let's explain step by step."}
]

show_messages(messages_cot)
_ = chat(messages_cot, temperature = 0.0)

STANDARD PROMPT


46


⏱️ 0.58s | 2 chars
PROMPT WITH COT


To calculate the total amount Ajay pays after using a 20% discount coupon on his purchase of 5 shirts and 3 pairs of pants:

1. Calculate the cost of the shirts:
   - Each shirt costs $8.
   - He buys 5 shirts: \( 5 \times \$8 = \$40 \).

2. Calculate the cost of the pants:
   - Each pair of pants costs $10.
   - He buys 3 pairs of pants: \( 3 \times \$10 = \$30 \).

3. Add the total cost of shirts and pants to get the subtotal before discount:
   - Subtotal: \( \$40 + \$30 = \$70 \).

4. Calculate the amount of the discount (20% off):
   - 20% of $70 is \( 0.20 \times \$70 = \$14 \).

5. Subtract the discount from the subtotal to get the total amount paid:
   - Total: \( \$70 - \$14 = \$56 \).

Therefore, Ajay pays a total of $56 after using the 20% discount coupon.


⏱️ 3.21s | 777 chars


### Experiment 1B: Few-Shot CoT — Teaching a Reasoning Style

In [5]:
messages =[
    {"role": "system", "content": "You are a helpful math assistant. "
    "Respond with step by step. "
    "State the final answer."},

    {"role": "user", "content": "A train travels 60 km/h for 2.5 hours. How far does it travel?"},
    {"role": "assistant", "content": (
        "Step 1: Identify given values. \n"
        "  Speed = 60 km/h, Time = 2.5 hours\n\n"
        "Step 2: Apply the formula, Distance = Speed × Time.\n"
        "  Distance = 60 × 2.5 = 150 km/h\n\n"
        "  **Final answer = 150 km/h**"
    )},

    {"role": "user", "content": "If 4 workers can build a wall in 16 days, how many days will 10 workers take?"},
    {"role": "assistant", "content": (
            "Step 1: Find the total work in worker-days.\n"
            "  Total work = 4 workers × 16 days = 64 worker-days\n\n"
            "Step 2: Divide by the new number of workers.\n"
            "  Days = 64 worker-days ÷ 10 workers = 6.4 days\n\n"
            "**Final Answer: 6.4 days**"
    )},

    {"role": "user", "content": problem}
]

print("=" * 60)
print("FEW SHOT")
print("=" * 60)

show_messages(messages)
_ = chat(messages, temperature = 0.0)

FEW SHOT


Step 1: Calculate the cost of the shirts.
   Cost of shirts = Number of shirts × Price per shirt
                  = 5 × $8
                  = $40

Step 2: Calculate the cost of the pants.
   Cost of pants = Number of pants × Price per pant
                 = 3 × $10
                 = $30

Step 3: Add the costs together to find the total before discount.
   Total cost before discount = Cost of shirts + Cost of pants
                            = $40 + $30
                            = $70

Step 4: Calculate the amount of the discount.
   Discount percentage = 20%
   Amount of discount = Total cost × (Discount percentage / 100)
                      = $70 × (20 / 100)
                      = $70 × 0.2
                      = $14

Step 5: Subtract the discount from the total cost to find the final amount paid.
   Final amount paid = Total cost before discount - Amount of discount
                    = $70 - $14
                    = $56

**Final Answer:** Ajay pays a total of $56.


⏱️ 7.05s | 995 chars
